# Notebook 04 — Ablation Study Results

Phân tích và trực quan hóa kết quả 4 ablation steps sau khi train trên Kaggle T4.

**Yêu cầu:** Copy các file sau từ Kaggle về thư mục tương ứng:
```
results/step1_baseline.json
results/step2_coc_shrink.json
results/step3_coc_hamming.json
results/step4_hbcc_full.json
experiments/resnet18_cifar10/log.csv
experiments/coc_baseline_cifar10/log.csv
experiments/hbcc_cifar10/log.csv  (steps 3 & 4 nếu có)
```

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Đường dẫn gốc project
ROOT = os.path.abspath('..')
RESULTS_DIR     = os.path.join(ROOT, 'results')
EXPERIMENTS_DIR = os.path.join(ROOT, 'experiments')
FIGURES_DIR     = os.path.join(ROOT, 'paper', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Setup OK. ROOT:', ROOT)

## 1. Load kết quả từ JSON

In [ ]:
STEP_META = [
    ('step1_baseline',   'Step 1 — ResNet-18',   'resnet18_cifar10',         '#4C72B0'),
    ('step2_coc_shrink', 'Step 2 — CoC+Shrink',  'coc_baseline_cifar10',     '#DD8452'),
    ('step3_coc_hamming','Step 3 — CoC+Hamming', 'coc_baseline_cifar10',     '#55A868'),
    ('step4_hbcc_full',  'Step 4 — HBCC Full',   'hbcc_cifar10',             '#C44E52'),
]

results = {}
for key, label, exp_dir, color in STEP_META:
    path = os.path.join(RESULTS_DIR, f'{key}.json')
    if os.path.exists(path):
        with open(path) as f:
            r = json.load(f)
        r['_label'] = label
        r['_exp_dir'] = exp_dir
        r['_color'] = color
        results[key] = r
        print(f'  OK: {label:30s} Acc={r.get("best_val_acc1", 0):.2f}%')
    else:
        print(f'  MISSING: {path}')

print(f'\nLoaded {len(results)}/4 result files')

## 2. Bảng so sánh Ablation

In [ ]:
rows = []
for key, label, _, _ in STEP_META:
    if key not in results:
        continue
    r = results[key]
    flags = r.get('ablation_flags', {})
    rows.append({
        'Step':       r.get('_label', key),
        'Model':      r.get('model', '—'),
        'PointShrink': '✓' if flags.get('use_point_shrink') else '✗',
        'Hamming':     '✓' if flags.get('use_hamming')      else '✗',
        'Bottleneck':  '✓' if flags.get('use_linear_bottleneck') else '✗',
        'ChShuffle':   '✓' if flags.get('use_channel_shuffle') else '✗',
        'Val Acc@1 (%)': round(r.get('best_val_acc1', 0), 2),
        'Params (M)':    round(r.get('total_params', 0) / 1e6, 2),
        'FLOPs (M)':     round(r.get('flops', 0) / 1e6, 2),
        'Throughput (img/s)': round(r.get('throughput', 0), 0),
        'Latency (ms)': round(r.get('latency_ms', 0), 2),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Lưu ra CSV để import Word/Excel
csv_out = os.path.join(RESULTS_DIR, 'ablation_table.csv')
df.to_csv(csv_out, index=False)
print(f'\nSaved: {csv_out}')

## 3. Training Curves — Val Acc@1 theo epoch

Đọc từ `experiments/<run_name>/log.csv` (ghi tự động trong lúc train).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

loaded_any = False
for key, label, exp_dir, color in STEP_META:
    csv_path = os.path.join(EXPERIMENTS_DIR, exp_dir, 'log.csv')
    if not os.path.exists(csv_path):
        # Thử thêm suffix _stepN nếu có nhiều runs
        alt = os.path.join(EXPERIMENTS_DIR, f'{exp_dir}_{key}', 'log.csv')
        if os.path.exists(alt):
            csv_path = alt
        else:
            print(f'  [skip] {label}: {csv_path} not found')
            continue

    df_log = pd.read_csv(csv_path)
    loaded_any = True

    # Val Acc
    axes[0].plot(df_log['epoch'], df_log['val_acc1'],
                 label=label, color=color, linewidth=1.5)
    # Train Loss
    axes[1].plot(df_log['epoch'], df_log['train_loss'],
                 label=label, color=color, linewidth=1.5)

if loaded_any:
    axes[0].set_title('Validation Accuracy@1')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Acc@1 (%)')
    axes[0].legend(fontsize=9)

    axes[1].set_title('Training Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend(fontsize=9)

    plt.tight_layout()
    fig_path = os.path.join(FIGURES_DIR, '04_training_curves.png')
    plt.savefig(fig_path, bbox_inches='tight')
    print(f'Saved: {fig_path}')
    plt.show()
else:
    print('Chưa có log.csv nào. Copy từ Kaggle về experiments/ trước.')
    plt.close()

## 4. Accuracy vs Params — Efficiency Plot

In [ ]:
if len(results) >= 2:
    fig, ax = plt.subplots(figsize=(8, 5))

    for key, label, _, color in STEP_META:
        if key not in results:
            continue
        r = results[key]
        params_m = r.get('total_params', 0) / 1e6
        acc      = r.get('best_val_acc1', 0)
        tp       = r.get('throughput', 1)

        # Kích thước dot = throughput (tương đối)
        ax.scatter(params_m, acc, s=200, color=color, zorder=5, edgecolors='white', linewidth=1.5)
        ax.annotate(label, (params_m, acc),
                    textcoords='offset points', xytext=(8, 4),
                    fontsize=9, color=color, fontweight='bold')

    ax.set_xlabel('Parameters (M)', fontsize=12)
    ax.set_ylabel('Val Acc@1 (%)', fontsize=12)
    ax.set_title('Accuracy vs Parameters Tradeoff', fontsize=13, fontweight='bold')

    # Chú thích góc
    ax.text(0.98, 0.04, 'Trên-trái = tốt hơn (cao hơn, ít params hơn)',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8, color='gray', style='italic')

    plt.tight_layout()
    fig_path = os.path.join(FIGURES_DIR, '04_efficiency_plot.png')
    plt.savefig(fig_path, bbox_inches='tight')
    print(f'Saved: {fig_path}')
    plt.show()
else:
    print('Cần ít nhất 2 kết quả để vẽ efficiency plot.')

## 5. Throughput — Tốc độ inference (img/s)

In [ ]:
if results:
    labels_bar = [r['_label'] for r in results.values()]
    throughputs = [r.get('throughput', 0) for r in results.values()]
    colors_bar  = [r['_color'] for r in results.values()]

    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(labels_bar, throughputs, color=colors_bar, edgecolor='white', height=0.5)

    # Label giá trị
    for bar, val in zip(bars, throughputs):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                f'{val:.0f} img/s', va='center', fontsize=10)

    ax.set_xlabel('Throughput (images/second)', fontsize=12)
    ax.set_title('Inference Throughput Comparison', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    ax.set_xlim(0, max(throughputs) * 1.2 if throughputs else 100)

    plt.tight_layout()
    fig_path = os.path.join(FIGURES_DIR, '04_throughput.png')
    plt.savefig(fig_path, bbox_inches='tight')
    print(f'Saved: {fig_path}')
    plt.show()

## 6. Delta Analysis — Đóng góp của từng kỹ thuật

In [ ]:
# So sánh delta accuracy và delta params giữa các steps liên tiếp
steps_ordered = ['step1_baseline','step2_coc_shrink','step3_coc_hamming','step4_hbcc_full']
available = [s for s in steps_ordered if s in results]

if len(available) >= 2:
    print(f'{'Transition':<40} {'ΔAcc@1':>10} {'ΔParams (M)':>14} {'ΔThroughput':>14}')
    print('-' * 80)
    for i in range(len(available)-1):
        a, b = available[i], available[i+1]
        ra, rb = results[a], results[b]
        delta_acc    = rb.get('best_val_acc1', 0) - ra.get('best_val_acc1', 0)
        delta_params = (rb.get('total_params', 0) - ra.get('total_params', 0)) / 1e6
        delta_tp     = rb.get('throughput', 0) - ra.get('throughput', 0)
        sign_acc = '+' if delta_acc >= 0 else ''
        sign_p   = '+' if delta_params >= 0 else ''
        sign_tp  = '+' if delta_tp >= 0 else ''
        label = f"{ra['_label']} → {rb['_label']}"
        print(f'{label:<40} {sign_acc}{delta_acc:>9.2f}% '
              f'{sign_p}{delta_params:>13.2f}M '
              f'{sign_tp}{delta_tp:>12.0f}')
else:
    print('Cần ít nhất 2 kết quả để tính delta.')

## 7. Tổng kết & Kết luận

In [ ]:
if results:
    best_acc_key  = max(results, key=lambda k: results[k].get('best_val_acc1', 0))
    best_tp_key   = max(results, key=lambda k: results[k].get('throughput', 0))
    fewest_p_key  = min(results, key=lambda k: results[k].get('total_params', float('inf')))

    print('=== KẾT LUẬN ABLATION STUDY ===')
    print(f'Best Accuracy   : {results[best_acc_key]["_label"]} '
          f'({results[best_acc_key].get("best_val_acc1", 0):.2f}%)')
    print(f'Best Throughput : {results[best_tp_key]["_label"]} '
          f'({results[best_tp_key].get("throughput", 0):.0f} img/s)')
    print(f'Lightest Model  : {results[fewest_p_key]["_label"]} '
          f'({results[fewest_p_key].get("total_params", 0)/1e6:.2f}M params)')

    if 'step1_baseline' in results and 'step4_hbcc_full' in results:
        r1 = results['step1_baseline']
        r4 = results['step4_hbcc_full']
        acc_drop   = r1.get('best_val_acc1', 0) - r4.get('best_val_acc1', 0)
        param_drop = (r1.get('total_params', 1) / r4.get('total_params', 1))
        tp_gain    = (r4.get('throughput', 0) / max(r1.get('throughput', 1), 1))
        print(f'\nHBCC Full vs ResNet-18:')
        print(f'  Accuracy drop : {acc_drop:.2f}%')
        print(f'  Params ratio  : {param_drop:.1f}× lighter')
        print(f'  Throughput    : {tp_gain:.1f}× faster')

    print(f'\nFigures đã lưu vào: {FIGURES_DIR}')